## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-04-05-55-11 +0000,nypost,"California voters reject Prop 39, stricter vot...",https://nypost.com/2026/11/04/us-news/californ...
1,2026-11-04-05-54-25 +0000,nyt,"David Crowley, a Democrat, Will Be Wisconsin’s...",https://www.nytimes.com/2026/11/03/us/wisconsi...
2,2026-11-04-05-53-15 +0000,startribune,Democrat Josh Turek wins election to the US Se...,https://www.startribune.com/a-rare-open-senate...
3,2026-11-04-05-53-05 +0000,startribune,Democrat Annie Andrews wins US Senate election...,https://www.startribune.com/south-carolina-sen...
4,2026-11-04-05-52-05 +0000,bbc,Ecstatic scenes and historic moments: What's h...,https://www.bbc.co.uk/news/articles/cmr4y92rgd...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2439/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
49,race,50
20,senate,47
19,election,46
43,trump,43
22,democrats,42


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
2,2026-11-04-05-53-15 +0000,startribune,Democrat Josh Turek wins election to the US Se...,https://www.startribune.com/a-rare-open-senate...,282
151,2026-11-04-03-14-56 +0000,chitri,South Carolina Senate race: Democrat Annie And...,https://www.chicagotribune.com/2026/11/03/sout...,273
202,2026-11-04-00-55-30 +0000,chitri,North Carolina Senate race: Democrat Roy Coope...,https://www.chicagotribune.com/2026/11/03/nort...,266
171,2026-11-04-02-18-11 +0000,chitri,Texas Senate race: Democrat James Talarico bea...,https://www.chicagotribune.com/2026/11/03/texa...,254
243,2026-11-03-22-49-00 +0000,wsj,Republicans and Democrats face off in a high-s...,https://www.wsj.com/politics/policy/democrats-...,250


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
2,282,2026-11-04-05-53-15 +0000,startribune,Democrat Josh Turek wins election to the US Se...,https://www.startribune.com/a-rare-open-senate...
243,208,2026-11-03-22-49-00 +0000,wsj,Republicans and Democrats face off in a high-s...,https://www.wsj.com/politics/policy/democrats-...
202,185,2026-11-04-00-55-30 +0000,chitri,North Carolina Senate race: Democrat Roy Coope...,https://www.chicagotribune.com/2026/11/03/nort...
165,96,2026-11-04-02-35-53 +0000,chitri,Georgia governor race: Democrat Keisha Lance B...,https://www.chicagotribune.com/2026/11/03/geor...
98,85,2026-11-04-04-33-12 +0000,chitri,Lake County Sheriff John Idleburg declares vic...,https://www.chicagotribune.com/2026/11/03/lake...
196,81,2026-11-04-01-00-20 +0000,chitri,Voters usher in new era with state Rep. La Sha...,https://www.chicagotribune.com/2026/11/03/illi...
63,79,2026-11-04-05-07-00 +0000,wsj,President Trump said on Tuesday night that the...,https://www.wsj.com/politics/elections/trump-a...
238,78,2026-11-03-23-00-00 +0000,wsj,Democrats and Republicans are battling over th...,https://www.wsj.com/politics/elections/early-e...
432,77,2026-11-03-11-57-00 +0000,wsj,"Follow the WSJ’s Election Day live coverage, a...",https://www.wsj.com/politics/elections/your-ho...
152,60,2026-11-04-03-08-31 +0000,chitri,Illinois General Assembly incumbents see early...,https://www.chicagotribune.com/2026/11/03/illi...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
